# Forsteinrichtungsoperate — Pipeline

Transcribes 19th-century Bavarian/German handwritten forestry records using Gemini.

**Steps:**
1. Install dependencies
2. Enter your Gemini API key
3. Upload scan images
4. Run the pipeline
5. Download results

## 1 — Clone repo & install dependencies

In [ ]:
import os

# Clone the repo (skip if already present)
if not os.path.isdir('Forsteinrichtungsoperate'):
    !git clone https://github.com/Maelkolb/Forsteinrichtungsoperate.git

%cd Forsteinrichtungsoperate

!pip install -q google-genai Pillow numpy

## 2 — Set your Gemini API key

Get a key at [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey). 
In Colab you can also store it as a secret under the key icon on the left sidebar.

In [ ]:
import os

# Option A: Colab Secrets (recommended — key icon in left sidebar)
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')
except Exception:
    # Option B: paste directly
    os.environ['GOOGLE_API_KEY'] = 'PASTE_YOUR_KEY_HERE'
    print('API key set manually.')

## 3 — Upload scan images

In [ ]:
import os
from google.colab import files

os.makedirs('input', exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'input/{name}', 'wb') as f:
        f.write(data)

print(f'Uploaded {len(uploaded)} file(s) to input/')

## 4 — Run the pipeline

Adjust flags as needed. Common options:
- `--double-page` — if your scans are two-page spreads
- `--deskew` — for skewed scans (installs scipy)
- `--model gemini-3.1-pro-preview` — for highest accuracy
- `--workers 2` — reduce if you hit rate limits

In [ ]:
!python run.py \
    --input input/ \
    --output output/ \
    --model gemini-3-flash-preview \
    --workers 2 \
    --verbose

## 5 — Inspect results

In [ ]:
from pathlib import Path

md_files = sorted(Path('output/md').glob('*.md'))
print(f'Produced {len(md_files)} Markdown file(s):')
for p in md_files:
    print(' ', p.name)

# Preview the first result
if md_files:
    print('\n--- Preview:', md_files[0].name, '---')
    print(md_files[0].read_text()[:2000])

## 6 — Download results as a zip

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('forsteinrichtung_output', 'zip', 'output')
files.download('forsteinrichtung_output.zip')